# 🧪 Lab 05 — Geometry Autopsy: WKT vs WKB, Human vs Alien 👽🔬

Humans like this:

```text
POINT(-3.7038 40.4168)
```

Machines are perfectly happy with this:

```text
0101000000...
```

Welcome to the autopsy.

This lab follows one Madrid point across Spark 4.2's native spatial boundary:

```text
human-readable WKT idea
        ↓
ordinary WKB bytes
        ↓
ST_GeomFromWKB
        ↓
GEOMETRY(4326)
        ↓
ST_AsBinary
        ↓
WKB again
```

Then we stop being polite and attack the parser.

### 🎯 Mission objectives

We will prove that:

- the same WKB payload means very different things as `BINARY` and `GEOMETRY(4326)`;
- `ST_GeomFromWKB` moves WKB across Spark's native spatial type boundary;
- `ST_AsBinary` moves it back to WKB;
- a valid Point survives an exact NDR round trip;
- NDR (little-endian) and XDR (big-endian) can encode the same geometry with different bytes;
- stock Spark 4.2 exposes exactly the small native `ST_*` surface discussed in the article;
- Spark rejects truncated WKB, invalid byte order, infinities, malformed LineStrings, invalid Polygon rings, and invalid Geography coordinates;
- NaN has an important exception: an empty Point may use NaN coordinates, while NaN inside non-point structures is rejected.

> **Evidence boundary:** WKT is used here as the human-readable reference notation. Stock Spark 4.2's built-in spatial constructor API tested in this notebook is WKB-centric; we do not invent an `ST_GeomFromText` function that Spark does not provide.

## 0 — Pre-flight checks 🛰️

Target environment:

```text
PySpark  4.2.0
Spark    4.2.0
Java     17+
```

No Sedona. No Shapely. No GeoPandas.

We deliberately construct the tiny WKB samples ourselves with Python's standard `struct` module so every byte on the operating table is ours.

In [1]:
import sys, json, math, struct, warnings


warnings.filterwarnings(
    "ignore",
    message=r"PySpark does not yet fully support pandas >= 3\.0\.0.*",
    category=FutureWarning,
)

import pyspark
from pyspark.sql import SparkSession, functions as F

active = SparkSession.getActiveSession()
if active is not None:
    active.stop()

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("lab-05-geometry-autopsy-wkt-wkb-human-vs-alien")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

java_version = spark.sparkContext._jvm.java.lang.System.getProperty("java.version")

fingerprint = {
    "python": sys.version.split()[0],
    "pyspark": pyspark.__version__,
    "spark": spark.version,
    "java": java_version,
    "geospatial_enabled": spark.conf.get("spark.sql.geospatial.enabled"),
}

print("🚀 Runtime fingerprint")
print(json.dumps(fingerprint, indent=2))

assert pyspark.__version__ == "4.2.0"
assert spark.version == "4.2.0"
assert int(java_version.split(".")[0]) >= 17
assert fingerprint["geospatial_enabled"].lower() == "true"

print("\n✅ Alien containment chamber online.")

🚀 Runtime fingerprint
{
  "python": "3.14.0",
  "pyspark": "4.2.0",
  "spark": "4.2.0",
  "java": "17.0.19",
  "geospatial_enabled": "true"
}

✅ Alien containment chamber online.


# 1 — WKT for Humans, WKB for Aliens 👽

The human version:

```text
POINT(1 2)
```

is **Well-Known Text (WKT)**.

The same geometry can be represented as WKB.

For a little-endian 2D Point, the bytes encode roughly:

```text
byte order
geometry type
X
Y
```

We are not building a full WKB library here. We only need enough machinery to create deterministic Points, LineStrings, and Polygons for the experiments.

In [2]:
def _prefix(endian):
    if endian == "NDR":
        return 1, "<"
    if endian == "XDR":
        return 0, ">"
    raise ValueError("endian must be NDR or XDR")

def wkb_point(x, y, endian="NDR"):
    byte_order, fmt = _prefix(endian)
    return struct.pack(f"{fmt}BIdd", byte_order, 1, float(x), float(y))

def wkb_linestring(points, endian="NDR"):
    byte_order, fmt = _prefix(endian)
    payload = struct.pack(f"{fmt}BI", byte_order, 2)
    payload += struct.pack(f"{fmt}I", len(points))
    for x, y in points:
        payload += struct.pack(f"{fmt}dd", float(x), float(y))
    return payload

def wkb_polygon(rings, endian="NDR"):
    byte_order, fmt = _prefix(endian)
    payload = struct.pack(f"{fmt}BI", byte_order, 3)
    payload += struct.pack(f"{fmt}I", len(rings))
    for ring in rings:
        payload += struct.pack(f"{fmt}I", len(ring))
        for x, y in ring:
            payload += struct.pack(f"{fmt}dd", float(x), float(y))
    return payload

def decode_wkb_point(wkb):
    byte_order = wkb[0]
    fmt = "<" if byte_order == 1 else ">"
    _, geom_type, x, y = struct.unpack(f"{fmt}BIdd", wkb)
    assert geom_type == 1
    return x, y

point_1_2_ndr = wkb_point(1, 2, "NDR")
point_1_2_xdr = wkb_point(1, 2, "XDR")

print("👽 POINT(1 2)")
print(f"  ├─ WKT human reference : POINT(1 2)")
print(f"  ├─ WKB NDR hex         : {point_1_2_ndr.hex().upper()}")
print(f"  └─ WKB XDR hex         : {point_1_2_xdr.hex().upper()}")

assert point_1_2_ndr.hex().upper() == "0101000000000000000000F03F0000000000000040"
assert point_1_2_xdr.hex().upper() == "00000000013FF00000000000004000000000000000"

👽 POINT(1 2)
  ├─ WKT human reference : POINT(1 2)
  ├─ WKB NDR hex         : 0101000000000000000000F03F0000000000000040
  └─ WKB XDR hex         : 00000000013FF00000000000004000000000000000


# 2 — Same Bytes, Completely Different Contract 🧬

Now we use Madrid:

```text
POINT(-3.7038 40.4168)
```

First we put the WKB into a normal DataFrame.

Spark sees:

```text
BINARY
```

Nothing in that schema says:

```text
this is a Point
this is spatial
this uses SRID 4326
```

Then we pass the exact same payload through:

```python
ST_GeomFromWKB(wkb, 4326)
```

and inspect what Spark knows afterward.

In [3]:
madrid_wkt = "POINT(-3.7038 40.4168)"
madrid_wkb = wkb_point(-3.7038, 40.4168, "NDR")

raw = spark.createDataFrame(
    [(1, madrid_wkt, madrid_wkb)],
    ["id", "wkt_reference", "payload"],
)

print("🧬 BEFORE: ordinary payload")
raw.printSchema()

raw_type = raw.select(
    F.expr("typeof(payload)").alias("payload_type")
).first().payload_type

native = raw.select(
    "id",
    "wkt_reference",
    "payload",
    F.st_geomfromwkb("payload", 4326).alias("geom"),
)

print("\n🌍 AFTER: native spatial value")
native.printSchema()

native_row = native.select(
    F.expr("typeof(geom)").alias("geom_type"),
    F.st_srid("geom").alias("srid"),
    F.st_asbinary("geom").alias("roundtrip_wkb"),
).first()

print("\n🔬 CONTRACT COMPARISON")
print(f"  ├─ raw Spark type     : {raw_type}")
print(f"  ├─ native Spark type  : {native_row.geom_type}")
print(f"  ├─ native SRID        : {native_row.srid}")
print(f"  └─ same WKB payload?  : {native_row.roundtrip_wkb == madrid_wkb}")

assert raw_type.lower() == "binary"
assert native_row.geom_type.lower() == "geometry(4326)"
assert native_row.srid == 4326
assert native_row.roundtrip_wkb == madrid_wkb

lab_results = {
    "raw_type": raw_type,
    "native_type": native_row.geom_type,
    "native_srid": native_row.srid,
    "ndr_roundtrip_exact": native_row.roundtrip_wkb == madrid_wkb,
}

🧬 BEFORE: ordinary payload
root
 |-- id: long (nullable = true)
 |-- wkt_reference: string (nullable = true)
 |-- payload: binary (nullable = true)


🌍 AFTER: native spatial value
root
 |-- id: long (nullable = true)
 |-- wkt_reference: string (nullable = true)
 |-- payload: binary (nullable = true)
 |-- geom: geometry(4326) (nullable = true)


🔬 CONTRACT COMPARISON
  ├─ raw Spark type     : binary
  ├─ native Spark type  : geometry(4326)
  ├─ native SRID        : 4326
  └─ same WKB payload?  : True


## The important part is not the bytes

Before:

```text
payload: BINARY
```

After:

```text
geom: GEOMETRY(4326)
```

The physical WKB can round-trip unchanged.

The logical contract is radically different.

`BINARY` tells Spark:

> bytes.

`GEOMETRY(4326)` tells Spark:

> a spatial value, with Geometry semantics, using one declared spatial reference.

That is the boundary this section cares about.

## 🧬 Same Bytes, Two Contracts

```text
                    SAME WKB BYTES
                         │
              ┌──────────┴──────────┐
              │                     │
           BINARY              GEOMETRY(4326)
              │                     │
      "I have bytes."        "I am spatial."
                             "I am Geometry."
                             "My SRID is 4326."
                             "Validate me as spatial data."
```

Nothing magical happened to Madrid's payload.

What changed was **what Spark is allowed to know and enforce about it**.

# 3 — How Many Native Spatial Buttons Are There? 🔘

Let's ask the actual Spark function catalog instead of trusting an article screenshot.

For stock Spark 4.2, the native spatial surface discussed here should be:

```text
ST_AsBinary
ST_GeogFromWKB
ST_GeomFromWKB
ST_SetSrid
ST_Srid
```

The theme is difficult to miss:

```text
construct
serialize
inspect SRID
change SRID
```

This is a representation/type boundary, not a complete GIS toolbox.

In [4]:
expected_st = {
    "st_asbinary",
    "st_geogfromwkb",
    "st_geomfromwkb",
    "st_setsrid",
    "st_srid",
}

registered_st = {
    f.name.lower()
    for f in spark.catalog.listFunctions()
    if f.name.lower().startswith("st_")
}

print("🔘 Stock Spark ST_* inventory")
for name in sorted(registered_st):
    print(f"  ├─ {name}")

missing = sorted(expected_st - registered_st)
extra = sorted(registered_st - expected_st)

print(f"\nExpected count : {len(expected_st)}")
print(f"Actual count   : {len(registered_st)}")
print(f"Missing        : {missing}")
print(f"Extra          : {extra}")

assert missing == []
assert extra == []

lab_results.update({
    "st_function_count": len(registered_st),
    "st_functions": sorted(registered_st),
})

🔘 Stock Spark ST_* inventory
  ├─ st_asbinary
  ├─ st_geogfromwkb
  ├─ st_geomfromwkb
  ├─ st_setsrid
  ├─ st_srid

Expected count : 5
Actual count   : 5
Missing        : []
Extra          : []


# 4 — NDR vs XDR: Same Geometry, Different Alien Dialect 👽↔️👽

WKB has a byte-order flag.

Spark 4.2's `ST_AsBinary` can emit:

```text
NDR → little-endian
XDR → big-endian
```

Two byte strings can therefore represent the **same geometry** while looking completely different in hex.

We will:

1. serialize Madrid as NDR;
2. serialize Madrid as XDR;
3. prove the bytes differ;
4. parse both;
5. canonicalize both back to NDR;
6. prove the canonical WKB is identical.

In [5]:
endian = native.select(
    F.st_asbinary("geom", "NDR").alias("ndr"),
    F.st_asbinary("geom", "XDR").alias("xdr"),
).first()

print("👽 WKB BYTE ORDER")
print(f"  ├─ NDR : {endian.ndr.hex().upper()}")
print(f"  └─ XDR : {endian.xdr.hex().upper()}")
print(f"\nRaw bytes identical? {endian.ndr == endian.xdr}")

assert endian.ndr != endian.xdr
assert endian.ndr[0] == 1
assert endian.xdr[0] == 0

both = spark.createDataFrame(
    [
        ("NDR", endian.ndr),
        ("XDR", endian.xdr),
    ],
    ["encoding", "wkb"],
).withColumn(
    "geom",
    F.st_geomfromwkb("wkb", 4326),
)

canonical = both.select(
    "encoding",
    F.st_asbinary("geom", "NDR").alias("canonical_ndr"),
).orderBy("encoding").collect()

canonical_payloads = [row.canonical_ndr for row in canonical]

print("\n🧬 Canonicalized back to NDR")
for row in canonical:
    print(f"  ├─ {row.encoding}: {row.canonical_ndr.hex().upper()}")

assert canonical_payloads[0] == canonical_payloads[1] == madrid_wkb

lab_results.update({
    "ndr_xdr_raw_different": endian.ndr != endian.xdr,
    "ndr_xdr_same_after_canonicalization": canonical_payloads[0] == canonical_payloads[1],
})

👽 WKB BYTE ORDER
  ├─ NDR : 0101000000FE65F7E461A10DC0857CD0B359354440
  └─ XDR : 0000000001C00DA161E4F765FE40443559B3D07C85

Raw bytes identical? False

🧬 Canonicalized back to NDR
  ├─ NDR: 0101000000FE65F7E461A10DC0857CD0B359354440
  ├─ XDR: 0101000000FE65F7E461A10DC0857CD0B359354440


# 5 — Geometry Autopsy: Attack the Parser 🔪👽

Now we deliberately manufacture bad WKB.

The cases are chosen to match Spark's documented WKB validation rules:

```text
truncated Point
invalid byte order
Infinity in a Point
LineString with only one point
NaN inside a LineString
unclosed Polygon ring
```

And one important control case:

```text
POINT(NaN NaN)
```

Spark allows NaN for an empty Point. So the intellectually honest rule is **not** “NaN is always invalid.”

It is closer to:

```text
empty Point NaN       → allowed
NaN in Line/Polygon   → rejected
Infinity anywhere     → rejected
```

Let's see what this actual runtime does.

In [6]:
def parse_geometry_case(name, payload, srid=4326):
    """
    Force Spark to parse one WKB payload as Geometry.
    Returns a small evidence dict; expected failures are captured, not thrown.
    """
    df = spark.createDataFrame([(name, payload)], ["case", "wkb"])

    spark.sparkContext.setLogLevel("OFF")
    try:
        row = (
            df.select(
                "case",
                F.st_geomfromwkb("wkb", srid).alias("geom"),
            )
            .select(
                "case",
                F.expr("typeof(geom)").alias("spark_type"),
                F.st_srid("geom").alias("srid"),
                F.st_asbinary("geom").alias("roundtrip"),
            )
            .first()
        )
        return {
            "case": name,
            "accepted": True,
            "spark_type": row.spark_type,
            "srid": row.srid,
            "error_type": None,
            "error_class": None,
            "error": None,
        }
    except Exception as exc:
        lines = [line.strip() for line in str(exc).splitlines() if line.strip()]
        error_class = None
        get_error_class = getattr(exc, "getErrorClass", None)
        if callable(get_error_class):
            try:
                error_class = get_error_class()
            except Exception:
                error_class = None
        if error_class is None and lines:
            import re
            match = re.search(r"\[([A-Z0-9_]+)\]", lines[0])
            if match:
                error_class = match.group(1)

        return {
            "case": name,
            "accepted": False,
            "spark_type": None,
            "srid": None,
            "error_type": type(exc).__name__,
            "error_class": error_class,
            "error": lines[0] if lines else repr(exc),
        }
    finally:
        spark.sparkContext.setLogLevel("ERROR")

valid_point = wkb_point(1, 2)

attacks = [
    ("valid Point control", valid_point, True),
    ("truncated Point", valid_point[:-8], False),
    ("invalid byte order", bytes([2]) + valid_point[1:], False),
    ("Point with Infinity", wkb_point(math.inf, 2), False),

    # Important nuance: NaN/NaN is Spark's representation of an empty Point.
    ("empty Point with NaN", wkb_point(math.nan, math.nan), True),

    ("LineString with one point", wkb_linestring([(0, 0)]), False),
    (
        "LineString containing NaN",
        wkb_linestring([(0, 0), (math.nan, 1)]),
        False,
    ),
    (
        "unclosed Polygon ring",
        wkb_polygon([[
            (0, 0),
            (10, 0),
            (10, 10),
            (0, 10),  # deliberately does NOT return to (0,0)
        ]]),
        False,
    ),
]

validation_results = []

for name, payload, expected_accept in attacks:
    result = parse_geometry_case(name, payload)
    result["expected_accept"] = expected_accept
    validation_results.append(result)

print("🔪 GEOMETRY PARSER AUTOPSY")
print("-" * 100)
for r in validation_results:
    status = "✅ ACCEPT" if r["accepted"] else "💥 REJECT"
    expected = "accept" if r["expected_accept"] else "reject"
    print(f"{status:<10} | expected {expected:<6} | {r['case']}")
    if r["error"]:
        error_label = f"[{r['error_class']}] " if r.get("error_class") else ""
        print(f"             └─ {error_label}{r['error'][:220]}")

assert all(
    r["accepted"] == r["expected_accept"]
    for r in validation_results
), validation_results

lab_results["geometry_validation"] = {
    r["case"]: r["accepted"]
    for r in validation_results
}

🔪 GEOMETRY PARSER AUTOPSY
----------------------------------------------------------------------------------------------------
✅ ACCEPT   | expected accept | valid Point control
💥 REJECT   | expected reject | truncated Point
             └─ [WKB_PARSE_ERROR] Error parsing WKB: Unexpected end of WKB buffer at position 13 SQLSTATE: 22023
💥 REJECT   | expected reject | invalid byte order
             └─ [WKB_PARSE_ERROR] Error parsing WKB: Invalid byte order 2 at position 0 SQLSTATE: 22023
💥 REJECT   | expected reject | Point with Infinity
             └─ [WKB_PARSE_ERROR] Error parsing WKB: Invalid coordinate value found at position 5 SQLSTATE: 22023
✅ ACCEPT   | expected accept | empty Point with NaN
💥 REJECT   | expected reject | LineString with one point
             └─ [WKB_PARSE_ERROR] Error parsing WKB: Too few points in linestring at position 5 SQLSTATE: 22023
💥 REJECT   | expected reject | LineString containing NaN
             └─ [WKB_PARSE_ERROR] Error parsing WKB: Invalid coor

## 🩺 Autopsy Report — This Actual Run

| Case | Result | Evidence from Spark |
|---|---|---|
| valid Point control | ✅ ACCEPT | valid WKB |
| truncated Point | 💥 REJECT | `[WKB_PARSE_ERROR]` |
| invalid byte order | 💥 REJECT | `[WKB_PARSE_ERROR]` |
| Point with Infinity | 💥 REJECT | `[WKB_PARSE_ERROR]` |
| empty Point with NaN | ✅ ACCEPT | valid empty-Point representation |
| LineString with one point | 💥 REJECT | `[WKB_PARSE_ERROR]` |
| LineString containing NaN | 💥 REJECT | `[WKB_PARSE_ERROR]` |
| unclosed Polygon ring | 💥 REJECT | `[WKB_PARSE_ERROR]` |

The important pattern is not simply “bad bytes fail.” Spark is enforcing **different classes of spatial validity**:

```text
binary structure
→ byte order, truncation

geometry structure
→ minimum LineString size, closed Polygon rings

coordinate validity
→ Infinity / contextual NaN rules

geographic validity
→ longitude / latitude bounds for GEOGRAPHY
```

That is an engine boundary, not a passive byte pipe.

# 6 — Geography Adds Another Guardrail 🌎🚧

The exact same WKB can be structurally valid Geometry but invalid Geography.

Why?

Because `GEOGRAPHY(4326)` adds longitude/latitude bounds:

```text
longitude ∈ [-180, 180]
latitude  ∈ [ -90,  90]
```

Let's use:

```text
POINT(200 95)
```

Perfectly ordinary numbers on graph paper.

Completely illegal longitude/latitude.

In [7]:
out_of_bounds_wkb = wkb_point(200, 95)

bounds_df = spark.createDataFrame(
    [(out_of_bounds_wkb,)],
    ["wkb"],
)

geometry_bounds_row = (
    bounds_df
    .select(F.st_geomfromwkb("wkb", 4326).alias("geom"))
    .select(
        F.expr("typeof(geom)").alias("spark_type"),
        F.st_srid("geom").alias("srid"),
    )
    .first()
)

geography_error = None
spark.sparkContext.setLogLevel("OFF")
try:
    bounds_df.select(
        F.st_geogfromwkb("wkb").alias("geog")
    ).collect()
except Exception as exc:
    geography_error = exc
finally:
    spark.sparkContext.setLogLevel("ERROR")

print("🌎 SAME WKB, DIFFERENT VALIDATION")
print(f"  ├─ GEOMETRY(4326) accepted : {geometry_bounds_row is not None}")
print(f"  ├─ Geometry type           : {geometry_bounds_row.spark_type}")
print(f"  └─ GEOGRAPHY(4326) accepted: {geography_error is None}")

if geography_error is not None:
    lines = [line.strip() for line in str(geography_error).splitlines() if line.strip()]
    print(f"      error: {(lines[0] if lines else repr(geography_error))[:240]}")

assert geometry_bounds_row.spark_type.lower() == "geometry(4326)"
assert geometry_bounds_row.srid == 4326
assert geography_error is not None

lab_results.update({
    "geometry_accepts_200_95": True,
    "geography_rejects_200_95": True,
    "geography_bounds_error_type": type(geography_error).__name__,
})

🌎 SAME WKB, DIFFERENT VALIDATION
  ├─ GEOMETRY(4326) accepted : True
  ├─ Geometry type           : geometry(4326)
  └─ GEOGRAPHY(4326) accepted: False
      error: [WKB_PARSE_ERROR] Error parsing WKB: Invalid coordinate value found at position 5 SQLSTATE: 22023


# 7 — What Moved Into the Engine Boundary? 🧠

This is the actual architectural result.

With raw `BINARY`, Spark can carry the bytes around, but the schema itself does not say:

```text
this is spatial
this is Geometry
this uses SRID 4326
```

After `ST_GeomFromWKB`, Spark has a native spatial value and the parser enforces spatial rules while constructing it.

Our autopsy covers three different classes of knowledge:

```text
representation
→ WKB parsing / serialization / byte order

type contract
→ GEOMETRY(4326), SRID inspection

validation
→ malformed structures, coordinate constraints
```

That is much more than “Spark can read some bytes.”

# 📊 Post-Lab Analysis — Let the Bodies Speak 👽🔬

The next cell builds its conclusions from this runtime's captured evidence.

If Spark accepts a malformed LineString, rejects an empty Point, or suddenly ships another stock `ST_*` function, an assertion above should stop the mission before this cell gets to write fan fiction.

In [8]:
from IPython.display import Markdown, display

accepted_cases = [
    name for name, accepted in lab_results["geometry_validation"].items()
    if accepted
]
rejected_cases = [
    name for name, accepted in lab_results["geometry_validation"].items()
    if not accepted
]

analysis = f"""
# 📊 Post-Lab Analysis: The Alien Was Mostly Just a Type Contract

We began with exactly the same Madrid WKB payload.

As an ordinary DataFrame column Spark saw:

**`{lab_results['raw_type']}`**

After `ST_GeomFromWKB(..., 4326)` it saw:

**`{lab_results['native_type']}`**

with SRID:

**`{lab_results['native_srid']}`**

and the default NDR WKB round trip was exact:

**{lab_results['ndr_roundtrip_exact']}**

### 1. Same Payload, Different Meaning

That is the core result.

The bytes can be identical while the logical contract changes from:

```text
BINARY
```

to:

```text
GEOMETRY(4326)
```

Spark has moved from “opaque bytes” to “native spatial value with an SRID contract.”

### 2. The Stock Native API Really Is Small

This runtime registered:

```text
{chr(10).join(lab_results['st_functions'])}
```

Total:

**{lab_results['st_function_count']}**

The native surface is therefore concentrated on construction, serialization, and SRID semantics rather than a broad GIS algorithm catalog.

### 3. WKB Bytes Are Not a Unique Visual Fingerprint

NDR and XDR raw WKB differed:

**{lab_results['ndr_xdr_raw_different']}**

But after both were parsed and canonicalized back to NDR, they matched:

**{lab_results['ndr_xdr_same_after_canonicalization']}**

So byte-for-byte inequality does **not** necessarily mean geometric inequality. Sometimes the alien is merely speaking big-endian.

### 4. The Parser Is an Actual Validation Boundary

Accepted Geometry cases:

```text
{chr(10).join(accepted_cases)}
```

Rejected Geometry cases:

```text
{chr(10).join(rejected_cases)}
```

The important NaN nuance survived the autopsy:

```text
empty Point with NaN       → {lab_results['geometry_validation']['empty Point with NaN']}
LineString containing NaN  → {lab_results['geometry_validation']['LineString containing NaN']}
```

NaN is therefore not universally illegal. Context matters.

### 5. Geography Adds Geographic Validation

`POINT(200 95)` was accepted as `GEOMETRY(4326)`:

**{lab_results['geometry_accepts_200_95']}**

but rejected as Geography:

**{lab_results['geography_rejects_200_95']}**

That is another reminder that SRID metadata and spatial type semantics work together.

> ## 🚀 Mission Verdict
> Spark 4.2's native spatial boundary is not merely a WKB decoder.
>
> It turns opaque `BINARY` into a typed spatial value, attaches SRID semantics, controls serialization, understands byte order, and rejects multiple classes of malformed or semantically illegal spatial input.
>
> **Same bytes. Much stronger contract.**
"""

display(Markdown(analysis))


# 📊 Post-Lab Analysis: The Alien Was Mostly Just a Type Contract

We began with exactly the same Madrid WKB payload.

As an ordinary DataFrame column Spark saw:

**`binary`**

After `ST_GeomFromWKB(..., 4326)` it saw:

**`geometry(4326)`**

with SRID:

**`4326`**

and the default NDR WKB round trip was exact:

**True**

### 1. Same Payload, Different Meaning

That is the core result.

The bytes can be identical while the logical contract changes from:

```text
BINARY
```

to:

```text
GEOMETRY(4326)
```

Spark has moved from “opaque bytes” to “native spatial value with an SRID contract.”

### 2. The Stock Native API Really Is Small

This runtime registered:

```text
st_asbinary
st_geogfromwkb
st_geomfromwkb
st_setsrid
st_srid
```

Total:

**5**

The native surface is therefore concentrated on construction, serialization, and SRID semantics rather than a broad GIS algorithm catalog.

### 3. WKB Bytes Are Not a Unique Visual Fingerprint

NDR and XDR raw WKB differed:

**True**

But after both were parsed and canonicalized back to NDR, they matched:

**True**

So byte-for-byte inequality does **not** necessarily mean geometric inequality. Sometimes the alien is merely speaking big-endian.

### 4. The Parser Is an Actual Validation Boundary

Accepted Geometry cases:

```text
valid Point control
empty Point with NaN
```

Rejected Geometry cases:

```text
truncated Point
invalid byte order
Point with Infinity
LineString with one point
LineString containing NaN
unclosed Polygon ring
```

The important NaN nuance survived the autopsy:

```text
empty Point with NaN       → True
LineString containing NaN  → False
```

NaN is therefore not universally illegal. Context matters.

### 5. Geography Adds Geographic Validation

`POINT(200 95)` was accepted as `GEOMETRY(4326)`:

**True**

but rejected as Geography:

**True**

That is another reminder that SRID metadata and spatial type semantics work together.

> ## 🚀 Mission Verdict
> Spark 4.2's native spatial boundary is not merely a WKB decoder.
>
> It turns opaque `BINARY` into a typed spatial value, attaches SRID semantics, controls serialization, understands byte order, and rejects multiple classes of malformed or semantically illegal spatial input.
>
> **Same bytes. Much stronger contract.**


## ✅ What This Lab Actually Proves

```text
raw WKB column is ordinary BINARY                    ✅
ST_GeomFromWKB(...,4326) creates GEOMETRY(4326)     ✅
SRID becomes engine-visible                          ✅
valid Point survives exact NDR round trip            ✅
NDR and XDR can differ byte-for-byte                 ✅
NDR/XDR canonicalize to the same geometry payload   ✅
stock Spark 4.2 exposes the five discussed ST_* funcs ✅

truncated Point rejected                             ✅
invalid byte order rejected                          ✅
Infinity rejected                                    ✅
one-point LineString rejected                        ✅
NaN inside LineString rejected                       ✅
unclosed Polygon ring rejected                       ✅
empty Point encoded with NaN accepted                ✅
invalid Geography lon/lat rejected                   ✅
```

The important conclusion is not:

> Spark can parse WKB.

It is:

> **Spark now owns an engine-level spatial boundary where representation, validation, and SRID/type semantics meet.**

## 👽 Autopsy in One Screen

```text
WKB bytes
   │
   ├── BINARY
   │      └── opaque payload
   │
   └── ST_GeomFromWKB(..., 4326)
          │
          └── GEOMETRY(4326)
                 ├── spatial type
                 ├── SRID contract
                 ├── WKB serialization
                 └── parser validation
```

And the runtime evidence says:

```text
valid structure                     → ✅
truncated / malformed WKB           → 💥
invalid coordinate structure        → 💥
NDR vs XDR                          → different bytes, same geometry
GEOMETRY geographic-looking values  → Cartesian rules
GEOGRAPHY values                    → geographic bounds too
```

> **Same bytes. Much stronger contract.**

# 🛰️ Mission Handoff

We have opened the alien binary and survived.

Now we know:

```text
WKT
→ excellent for humans

WKB
→ compact machine representation

BINARY
→ opaque payload contract

GEOMETRY / GEOGRAPHY
→ native spatial contract
```

The next question is bigger:

> **What happens when this native spatial contract goes to disk at scale?**

Next mission: **Parquet orbit — spatial logical types, CRS metadata, and what survives storage.** 🛰️📦

---

## 📚 Primary references

- Apache Spark 4.2 — Geospatial types and WKB validation rules  
  https://spark.apache.org/docs/latest/sql-ref-geospatial-types.html

- Apache Spark 4.2 — built-in geospatial `ST_*` functions  
  https://spark.apache.org/docs/latest/sql-ref-functions-builtin.html

- PySpark 4.2 — `st_asbinary` and NDR/XDR serialization  
  https://spark.apache.org/docs/4.2.0/api/python/reference/pyspark.sql/api/pyspark.sql.functions.st_asbinary.html

This notebook deliberately tests Spark's runtime behavior rather than treating the documentation as an expected-output script.

In [9]:
spark.stop()
print("👽 Spark stopped. Alien geometry returned to containment.")

👽 Spark stopped. Alien geometry returned to containment.
